# サンプル: 週次レポートの使い方

> **このノートブックは使い方のサンプルです（実行結果は保存しない）。**
> 週次レポートの定期実行は CLI で行う: `uv run marketdata weekly-report --date 2026-10-21`

1. CLI と同じ処理をコードから呼ぶ（`WeeklyReport`）
2. データを一度取得して、図表を個別に作る・調整する
3. 下位の関数を直接使う（任意のユニバースのリターン、ツリーマップなど）

## 1. CLI と同じ処理をコードから呼ぶ

In [ ]:
import pandas as pd
from IPython.display import Image, display

from marketdata import PROJECT_ROOT, LSEGClient, load_universe
from marketdata.reports.weekly import WeeklyReport, WeeklyReportConfig

# 設定は configs/weekly_report.yaml。ノートブック上で一部だけ上書きすることもできる
config = WeeklyReportConfig.load().model_copy(update={"min_weight": 0.001})

report = WeeklyReport(
    base_date=pd.Timestamp("2026-10-21"),  # 基準日（水曜）。価格系は前日までの値を使う
    config=config,
    output_dir=PROJECT_ROOT / "outputs" / "sample",  # 本番の outputs/weekly と分ける
)
print(report.base_date, report.return_start, report.return_end)

In [ ]:
# 全図表を出力（LSEG に接続してデータを取得する）
outputs = report.run()
[p.name for p in outputs]

## 2. データを一度取得して、図表を個別に作る

`fetch` で取得したデータを使い回せるので、見た目を調整するときに便利。

In [ ]:
with LSEGClient() as client:
    data = report.fetch(client)

report.index_returns(data)  # 01, 02
report.us_yield_curve(data)  # 06
display(Image(report.output_dir / "02_sector_returns.png", width=900))

In [ ]:
# 取得済みデータはそのまま DataFrame として見られる
data.earnings_dates.head()

## 3. 下位の関数を直接使う

In [ ]:
from marketdata.plotting import period_returns, plot_returns, plot_treemap, save_png, short_name

# universe.yaml のグループの1ヶ月リターン
universe = load_universe()
with LSEGClient() as client:
    prices = client.get_instrument_history(
        universe["countries"], start="2026-09-01", end="2026-10-20"
    )

monthly = period_returns(prices, "2026-09-20", "2026-10-20")
fig = plot_returns(monthly, title="各国株式インデックス 1ヶ月リターン", subtitle="現地通貨建て")
fig

In [ ]:
# ツリーマップ: 任意の銘柄表（weight / return / label / 階層の列）から描ける
stocks = pd.DataFrame(
    {
        "country": ["韓国", "韓国", "台湾", "インド"],
        "sector": ["情報技術", "金融", "情報技術", "金融"],
        "label": [
            short_name(n)
            for n in [
                "Samsung Electronics Co Ltd",
                "KB Financial Group Inc",
                "Taiwan Semiconductor Manufacturing Co Ltd",
                "HDFC Bank Ltd",
            ]
        ],
        "weight": [0.04, 0.01, 0.08, 0.02],
        "return": [-0.03, 0.01, 0.02, -0.005],
    }
)
fig = plot_treemap(
    stocks, ["country", "sector"], title="ツリーマップのサンプル", subtitle="ダミーデータ"
)
save_png(fig, PROJECT_ROOT / "outputs" / "sample" / "treemap_sample.png")
fig